In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 3, Finished, Available, Finished, False)

In [2]:
df_silver = spark.sql("""
SELECT *
FROM silver_sales
""")

display(df_silver)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 81cdbce3-8e80-45cd-9d3f-b047b0e52a61)

In [3]:
dim_customer = df_silver.select(
    "CustomerID",
    "Country"
).dropDuplicates()


dim_customer = dim_customer.withColumn(
    "CustomerKey",
    monotonically_increasing_id()
)


dim_customer = dim_customer.select(
    "CustomerKey",
    "CustomerID",
    "Country"
)


display(dim_customer)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 70ed6d69-f44c-4a3f-9231-8ec51a773d34)

In [4]:
dim_product = df_silver.select(
    "StockCode",
    "Description"
).dropDuplicates()


dim_product = dim_product.withColumn(
    "ProductKey",
    monotonically_increasing_id()
)


dim_product = dim_product.select(
    "ProductKey",
    "StockCode",
    "Description"
)


display(dim_product)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 41df9163-063b-45c8-8087-251830983d4c)

In [5]:
dim_date = df_silver.select(
    to_date("InvoiceDate").alias("Date")
).dropDuplicates()


dim_date = dim_date.withColumn(
    "DateKey",
    date_format("Date","yyyyMMdd").cast("int")
)


dim_date = dim_date.withColumn(
    "Year",
    year("Date")
)


dim_date = dim_date.withColumn(
    "Month",
    month("Date")
)


dim_date = dim_date.withColumn(
    "MonthName",
    date_format("Date","MMMM")
)


dim_date = dim_date.withColumn(
    "Quarter",
    concat(
        lit("Q"),
        quarter("Date")
    )
)


dim_date = dim_date.withColumn(
    "MonthYear",
    date_format(col("Date"),"MMM yyyy")
)


dim_date = dim_date.withColumn(
    "MonthYearSort",
    date_format(col("Date"),"yyyyMM").cast("int")
)


dim_date = dim_date.withColumn(
    "WeekNumber",
    weekofyear("Date")
)


dim_date = dim_date.withColumn(
    "DayName",
    date_format(col("Date"),"EEEE")
)


dim_date = dim_date.withColumn(
    "WeekendFlag",
    when(
        dayofweek("Date").isin([1,7]),
        "Weekend"
    )
    .otherwise("Weekday")
)


display(dim_date)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 657493b9-1be2-443a-8d54-f1e10348dc01)

In [6]:
fact_sales = df_silver

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 8, Finished, Available, Finished, False)

In [7]:
fact_sales = fact_sales.join(
    dim_customer,
    "CustomerID",
    "left"
)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 9, Finished, Available, Finished, False)

In [8]:
fact_sales = fact_sales.join(
    dim_product,
    "StockCode",
    "left"
)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 10, Finished, Available, Finished, False)

In [9]:
fact_sales = fact_sales.withColumn(
    "Date",
    to_date("InvoiceDate")
)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 11, Finished, Available, Finished, False)

In [10]:
fact_sales = fact_sales.join(
    dim_date,
    "Date",
    "left"
)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 12, Finished, Available, Finished, False)

In [11]:
fact_sales = fact_sales.select(
    "Invoice",
    "ProductKey",
    "CustomerKey",
    "DateKey",
    "Quantity",
    "Price",
    "Revenue"
)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 13, Finished, Available, Finished, False)

In [12]:
fact_sales = fact_sales.withColumnRenamed(
    "Invoice",
    "InvoiceNo"
)


fact_sales = fact_sales.withColumnRenamed(
    "Price",
    "UnitPrice"
)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 14, Finished, Available, Finished, False)

In [13]:
fact_sales = fact_sales.withColumn(
    "SalesCategory",
    when(col("Revenue") >= 500, "High Value")
    .when(col("Revenue") >= 100, "Medium Value")
    .otherwise("Low Value")
)


fact_sales = fact_sales.withColumn(
    "QuantityCategory",
    when(col("Quantity") >= 50, "Bulk Order")
    .when(col("Quantity") >= 10, "Medium Order")
    .otherwise("Small Order")
)


fact_sales = fact_sales.withColumn(
    "ReturnFlag",
    when(col("Quantity") < 0, "Return")
    .otherwise("Sale")
)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 15, Finished, Available, Finished, False)

In [14]:
customer_metrics = fact_sales.groupBy(
    "CustomerKey"
).agg(
    sum("Revenue").alias("CustomerRevenue"),
    countDistinct("InvoiceNo").alias("CustomerOrderCount")
)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 16, Finished, Available, Finished, False)

In [15]:
dim_customer = dim_customer.join(
    customer_metrics,
    "CustomerKey",
    "left"
)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 17, Finished, Available, Finished, False)

In [16]:
dim_customer = dim_customer.withColumn(
    "CustomerSegment",

    when(col("CustomerRevenue") >= 5000,
         "VIP Customer")

    .when(col("CustomerRevenue") >= 1000,
          "Regular Customer")

    .otherwise(
        "Low Value Customer"
    )
)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 18, Finished, Available, Finished, False)

In [17]:
product_metrics = fact_sales.groupBy(
    "ProductKey"
).agg(
    sum("Revenue").alias("ProductRevenue"),
    sum("Quantity").alias("QuantitySold")
)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 19, Finished, Available, Finished, False)

In [18]:
dim_product = dim_product.join(
    product_metrics,
    "ProductKey",
    "left"
)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 20, Finished, Available, Finished, False)

In [19]:
dim_product = dim_product.withColumn(
    "ProductPerformance",

    when(col("ProductRevenue") >= 10000,
         "High Performer")

    .when(col("ProductRevenue") >= 1000,
          "Medium Performer")

    .otherwise(
        "Low Performer"
    )
)

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 21, Finished, Available, Finished, False)

In [20]:
dim_customer.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("dim_customer")

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 22, Finished, Available, Finished, False)

In [21]:
dim_product.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("dim_product")

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 23, Finished, Available, Finished, False)

In [22]:
dim_date.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("dim_date")

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 24, Finished, Available, Finished, False)

In [23]:
fact_sales.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("fact_sales")

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 25, Finished, Available, Finished, False)

In [24]:
fact_sales.count()

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 26, Finished, Available, Finished, False)

986804

In [25]:
fact_sales.select(
    sum("Revenue").alias("Total Revenue")
).show()

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 27, Finished, Available, Finished, False)

+--------------------+
|       Total Revenue|
+--------------------+
|2.2285633098000586E7|
+--------------------+



In [26]:
spark.sql("SHOW TABLES").show()

StatementMeta(, c1019927-8c76-439f-b266-a45b362e82b1, 28, Finished, Available, Finished, False)

+--------------------+------------+-----------+
|           namespace|   tableName|isTemporary|
+--------------------+------------+-----------+
|`Customer Sales A...|bronze_sales|      false|
|`Customer Sales A...|dim_customer|      false|
|`Customer Sales A...|    dim_date|      false|
|`Customer Sales A...| dim_product|      false|
|`Customer Sales A...|  fact_sales|      false|
|`Customer Sales A...|silver_sales|      false|
+--------------------+------------+-----------+

